# Logo Adjustment Notebook
This notebook uses Pillow and NumPy to remove the white background from `logo.jpg` and place it into a frame.

## Section 1: Import Required Libraries
Import image processing libraries such as Pillow and NumPy, plus IPython display for inline rendering.

In [ ]:
from PIL import Image, ImageDraw
import numpy as np
from IPython.display import display

logo_path = 'logo.jpg'
output_path = 'logo_transparent_framed.png'

## Section 2: Load Frame and Logo Images
Read the logo image from disk and create a frame image to place the logo into.

In [ ]:
logo = Image.open(logo_path).convert('RGBA')

frame_size = (420, 420)
frame = Image.new('RGBA', frame_size, (0, 0, 0, 0))
draw = ImageDraw.Draw(frame)

border_width = 18
corner_radius = 36
for offset in range(border_width):
    draw.rounded_rectangle(
        [offset, offset, frame_size[0] - offset - 1, frame_size[1] - offset - 1],
        radius=corner_radius,
        outline=(234, 179, 8, 220)
    )

inner = Image.new('RGBA', frame_size, (235, 230, 210, 10))
frame = Image.alpha_composite(frame, inner)

## Section 3: Remove White Background from Logo
Detect white pixels and use them to generate an alpha mask, making the background transparent.

In [ ]:
logo_data = np.array(logo)
white_threshold = 240
mask = np.all(logo_data[:, :, :3] >= white_threshold, axis=2)
alpha_channel = np.where(mask, 0, logo_data[:, :, 3])
logo_data[:, :, 3] = alpha_channel
logo_transparent = Image.fromarray(logo_data, mode='RGBA')

## Section 4: Resize and Position Logo
Scale the logo to fit within the frame and compute centered placement coordinates.

In [ ]:
max_logo_width = frame_size[0] - 120
max_logo_height = frame_size[1] - 120
logo_transparent.thumbnail((max_logo_width, max_logo_height), Image.LANCZOS)

x = (frame_size[0] - logo_transparent.width) // 2
y = (frame_size[1] - logo_transparent.height) // 2

## Section 5: Blend Logo into Frame
Overlay the processed logo onto the frame using its alpha channel.

In [ ]:
result = frame.copy()
result.paste(logo_transparent, (x, y), logo_transparent)

## Section 6: Save and Display Result
Save the final composed image and display it inline.

In [ ]:
result.save(output_path)
print(f'Saved result to {output_path}')
display(result)